# Cross-Architecture Ensemble — Mitral ECG Classification

In [ ]:
from __future__ import annotations
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, roc_auc_score, confusion_matrix

ARTIFACT_ROOT = Path("dl_artifacts")
N_FOLDS = 5
CLASS_NAMES = ["Control", "MR", "MS"]

# 7 model variants from the technical report
MODEL_REGISTRY = {
    # Ribeiro ResNet (PTB-XL pretrained, 36-ch)
    "rib_ptbxl_iv":  "ribeiro_resnet_ecg_intervals_tiffany_flat",
    "rib_ext":       "ribeiro_resnet_ecg_features_ext_tiffany_flat",
    # InceptionTime nf=64 (12-ch overlap-zeroed)
    "incep_iv":      "inceptiontime_ecg_intervals_tiffany_flat_nf64",
    "incep_ext":     "inceptiontime_ecg_features_ext_tiffany_flat_nf64",
    "multitask":     "inceptiontime_multitask_tiffany_flat",
    # CNN-BiLSTM-Attention (12-ch overlap-zeroed)
    "cnn_bilstm":    "cnn_bilstm_attention_tiffany_flat",
    "cnn_iv":        "cnn_bilstm_ecg_intervals_tiffany_flat",
}

ARCH_FAMILY = {
    "rib_ptbxl_iv": "Ribeiro", "rib_ext": "Ribeiro",
    "incep_iv": "InceptionTime", "incep_ext": "InceptionTime", "multitask": "InceptionTime",
    "cnn_bilstm": "CNN-BiLSTM", "cnn_iv": "CNN-BiLSTM",
}

print(f"Models registered: {len(MODEL_REGISTRY)}")
for name, adir in MODEL_REGISTRY.items():
    exists = (ARTIFACT_ROOT / adir / "fold_0" / "heldout_logits.npz").exists()
    print(f"  {name:15s} [{ARCH_FAMILY[name]:12s}]  {'OK' if exists else 'MISSING'}")

## 1. Load Heldout Predictions

In [ ]:
def load_heldout(model_name: str) -> tuple[np.ndarray, np.ndarray]:
    """Load and average heldout probabilities across 5 folds."""
    adir = ARTIFACT_ROOT / MODEL_REGISTRY[model_name]
    probs_list = []
    y_true = None
    for fold in range(N_FOLDS):
        d = np.load(adir / f"fold_{fold}" / "heldout_logits.npz")
        probs_list.append(d["probs"])
        if y_true is None:
            y_true = d["y_true"]
            stems  = d["stems"]
    return y_true, np.mean(probs_list, axis=0), stems


def load_oof_val(model_name: str) -> tuple[np.ndarray, np.ndarray]:
    """Load pooled OOF val predictions (for threshold tuning)."""
    adir = ARTIFACT_ROOT / MODEL_REGISTRY[model_name]
    all_y, all_probs = [], []
    for fold in range(N_FOLDS):
        p = adir / f"fold_{fold}" / "val_logits.npz"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        d = np.load(p)
        all_y.append(d["y_true"])
        all_probs.append(d["probs"])
    return np.concatenate(all_y), np.concatenate(all_probs)


# Load all models
heldout_cache = {}
y_true_ref = None

for name in MODEL_REGISTRY:
    try:
        yt, probs, stems = load_heldout(name)
        heldout_cache[name] = probs
        if y_true_ref is None:
            y_true_ref = yt
        else:
            assert np.array_equal(yt, y_true_ref)
        print(f"  {name:15s}: loaded ({probs.shape})")
    except Exception as e:
        print(f"  {name:15s}: FAILED ({e})")

print(f"\nHeldout set: N={len(y_true_ref)}")
print(f"Class distribution: {dict(zip(CLASS_NAMES, np.bincount(y_true_ref.astype(int))))}")

## 2. Single-Model Baselines

In [ ]:
def evaluate(y_true, probs, delta=None):
    """Compute metrics. Optional logit-offset delta for threshold tuning."""
    if delta is not None:
        log_p = np.log(np.clip(probs, 1e-9, 1.0)) + np.array(delta)
        preds = log_p.argmax(axis=1)
    else:
        preds = probs.argmax(axis=1)
    f1  = f1_score(y_true, preds, average="macro", zero_division=0)
    rec = recall_score(y_true, preds, average=None, zero_division=0, labels=[0,1,2])
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    except:
        auc = float("nan")
    return {"f1": f1, "auc": auc, "ctrl": rec[0], "mr": rec[1], "ms": rec[2]}


print(f"{'Model':<15s}  {'Family':<12s}  {'F1':>6s}  {'AUC':>6s}  {'Ctrl':>5s}  {'MR':>5s}  {'MS':>5s}")
print("-" * 72)

single_results = []
for name in sorted(heldout_cache, key=lambda n: -evaluate(y_true_ref, heldout_cache[n])["f1"]):
    m = evaluate(y_true_ref, heldout_cache[name])
    print(f"{name:<15s}  {ARCH_FAMILY[name]:<12s}  {m['f1']:.4f}  {m['auc']:.4f}  {m['ctrl']:.3f}  {m['mr']:.3f}  {m['ms']:.3f}")
    single_results.append({"model": name, "family": ARCH_FAMILY[name], **m})

df_single = pd.DataFrame(single_results).sort_values("f1", ascending=False)
print(f"\nBest single model: {df_single.iloc[0]['model']} (F1={df_single.iloc[0]['f1']:.4f})")

## 3. Hard Voting Ensemble

Setiap model melakukan argmax → label, lalu majority vote menentukan prediksi final.  
Threshold tuning **tidak bisa diterapkan** karena informasi probabilitas sudah hilang.

In [ ]:
def hard_vote(model_names: list[str]) -> np.ndarray:
    """Majority vote dari argmax masing-masing model."""
    votes = np.stack([heldout_cache[n].argmax(axis=1) for n in model_names], axis=0)  # (M, N)
    preds = np.zeros(votes.shape[1], dtype=int)
    for i in range(votes.shape[1]):
        counts = np.bincount(votes[:, i], minlength=3)
        preds[i] = counts.argmax()
    return preds


def hard_vote_metrics(y_true, model_names):
    preds = hard_vote(model_names)
    f1  = f1_score(y_true, preds, average="macro", zero_division=0)
    rec = recall_score(y_true, preds, average=None, zero_division=0, labels=[0,1,2])
    avg_probs = np.mean([heldout_cache[n] for n in model_names], axis=0)
    try:
        auc = roc_auc_score(y_true, avg_probs, multi_class="ovr", average="macro")
    except:
        auc = float("nan")
    return {"f1": f1, "auc": auc, "ctrl": rec[0], "mr": rec[1], "ms": rec[2]}


# Evaluate all 2-model combinations
available = sorted(heldout_cache.keys())
hard_results = []

for r in range(2, min(len(available), 4) + 1):  # 2, 3, 4 model combos
    for combo in combinations(available, r):
        m = hard_vote_metrics(y_true_ref, list(combo))
        families = set(ARCH_FAMILY[n] for n in combo)
        hard_results.append({
            "ensemble": "+".join(combo),
            "n_models": r,
            "cross_arch": len(families) > 1,
            "families": ",".join(sorted(families)),
            **m,
        })

df_hard = pd.DataFrame(hard_results).sort_values("f1", ascending=False)

print("=== Top 10 Hard Voting Ensembles ===")
print(f"{'Ensemble':<50s}  {'Cross':>5s}  {'F1':>6s}  {'AUC':>6s}  {'Ctrl':>5s}  {'MR':>5s}  {'MS':>5s}")
print("-" * 90)
for _, row in df_hard.head(10).iterrows():
    ca = "YES" if row["cross_arch"] else "no"
    print(f"{row['ensemble']:<50s}  {ca:>5s}  {row['f1']:.4f}  {row['auc']:.4f}  {row['ctrl']:.3f}  {row['mr']:.3f}  {row['ms']:.3f}")

## 4. Soft Voting Ensemble

Probabilitas dari masing-masing model di-rata-ratakan, lalu argmax.

In [ ]:
def soft_vote_probs(model_names: list[str]) -> np.ndarray:
    """Average probabilities across models."""
    return np.mean([heldout_cache[n] for n in model_names], axis=0)


soft_results = []

for r in range(2, min(len(available), 4) + 1):
    for combo in combinations(available, r):
        probs = soft_vote_probs(list(combo))
        m = evaluate(y_true_ref, probs)
        families = set(ARCH_FAMILY[n] for n in combo)
        soft_results.append({
            "ensemble": "+".join(combo),
            "n_models": r,
            "cross_arch": len(families) > 1,
            "families": ",".join(sorted(families)),
            **m,
        })

df_soft = pd.DataFrame(soft_results).sort_values("f1", ascending=False)

print("=== Top 10 Soft Voting Ensembles ===")
print(f"{'Ensemble':<50s}  {'Cross':>5s}  {'F1':>6s}  {'AUC':>6s}  {'Ctrl':>5s}  {'MR':>5s}  {'MS':>5s}")
print("-" * 90)
for _, row in df_soft.head(10).iterrows():
    ca = "YES" if row["cross_arch"] else "no"
    print(f"{row['ensemble']:<50s}  {ca:>5s}  {row['f1']:.4f}  {row['auc']:.4f}  {row['ctrl']:.3f}  {row['mr']:.3f}  {row['ms']:.3f}")

## 5. Soft Voting + Threshold Optimization

Logit-offset δ = (0, δ_MR, δ_MS) di-grid-search pada **OOF val predictions** (fair — heldout tidak digunakan untuk tuning).  
δ_Control = 0 (referensi tetap), δ_MR dan δ_MS bervariasi di [-1.5, +1.5].

In [ ]:
def find_best_delta(y_oof, probs_oof, grid_steps=41, search_range=1.5):
    """Grid search logit-offset delta pada OOF val predictions."""
    log_probs = np.log(np.clip(probs_oof, 1e-9, 1.0))
    grid = np.linspace(-search_range, search_range, grid_steps)
    best_f1, best_delta = -1.0, (0.0, 0.0, 0.0)
    for d1 in grid:
        for d2 in grid:
            delta = np.array([0.0, d1, d2], dtype=np.float32)
            preds = (log_probs + delta).argmax(axis=1)
            f1 = f1_score(y_oof, preds, average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1, best_delta = f1, (0.0, float(d1), float(d2))
    return best_delta, best_f1


def build_ensemble_oof(model_names):
    """Pool OOF val predictions, aligned by common stems."""
    per_model = {}
    for name in model_names:
        adir = ARTIFACT_ROOT / MODEL_REGISTRY[name]
        for fold in range(N_FOLDS):
            p = adir / f"fold_{fold}" / "val_logits.npz"
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")
            d = np.load(p)
            for s, prob, y in zip(d["stems"], d["probs"], d["y_true"]):
                per_model.setdefault(name, {})[s] = (prob, y)

    common = set(per_model[model_names[0]].keys())
    for name in model_names[1:]:
        common &= set(per_model[name].keys())
    common = sorted(common)

    y_true = np.array([per_model[model_names[0]][s][1] for s in common])
    avg_probs = np.mean(
        [np.array([per_model[n][s][0] for s in common]) for n in model_names],
        axis=0
    )
    return y_true, avg_probs


# Select top ensembles from soft voting to apply threshold optimization
TOP_ENSEMBLES = [
    ["rib_ext", "multitask"],
    ["rib_ext", "cnn_iv"],
    ["rib_ext", "incep_iv"],
    ["rib_ext", "multitask", "rib_ptbxl_iv", "incep_iv"],
    ["rib_ext", "incep_ext", "multitask", "rib_ptbxl_iv"],
    ["rib_ext", "rib_ptbxl_iv", "incep_iv", "cnn_bilstm"],
    ["rib_ext", "incep_ext", "multitask", "cnn_bilstm"],
    ["rib_ext", "multitask", "cnn_bilstm"],
]

thresh_results = []

for model_names in TOP_ENSEMBLES:
    ens_name = "+".join(model_names)
    try:
        # OOF val (for tuning)
        y_oof, probs_oof = build_ensemble_oof(model_names)
        # Heldout (for reporting)
        probs_held = soft_vote_probs(model_names)

        m_base = evaluate(y_true_ref, probs_held)
        best_delta, oof_f1 = find_best_delta(y_oof, probs_oof)
        m_tuned = evaluate(y_true_ref, probs_held, delta=best_delta)

        families = set(ARCH_FAMILY[n] for n in model_names)
        thresh_results.append({
            "ensemble": ens_name,
            "n_models": len(model_names),
            "cross_arch": len(families) > 1,
            "f1_base": m_base["f1"],
            "f1_tuned": m_tuned["f1"],
            "delta_str": f"({best_delta[0]:.3f}, {best_delta[1]:.3f}, {best_delta[2]:.3f})",
            "auc": m_tuned["auc"],
            "ctrl": m_tuned["ctrl"],
            "mr": m_tuned["mr"],
            "ms": m_tuned["ms"],
        })
        print(f"{ens_name:<50s}  base={m_base['f1']:.4f}  tuned={m_tuned['f1']:.4f}  δ={best_delta}")
    except Exception as e:
        print(f"{ens_name:<50s}  FAILED: {e}")

df_thresh = pd.DataFrame(thresh_results).sort_values("f1_tuned", ascending=False)

print("\n=== Soft Voting + Threshold (Top Results) ===")
print(f"{'Ensemble':<50s}  {'Base':>6s}  {'Tuned':>6s}  {'δ':>25s}  {'Ctrl':>5s}  {'MR':>5s}  {'MS':>5s}")
print("-" * 110)
for _, row in df_thresh.iterrows():
    print(f"{row['ensemble']:<50s}  {row['f1_base']:.4f}  {row['f1_tuned']:.4f}  {row['delta_str']:>25s}  {row['ctrl']:.3f}  {row['mr']:.3f}  {row['ms']:.3f}")

## 6. Cross-Arch vs Within-Arch Analysis

In [ ]:
print("=== Hard Voting: Cross-Arch vs Within-Arch ===")
for label, subset in [("Cross-arch", df_hard[df_hard["cross_arch"]]),
                       ("Within-arch", df_hard[~df_hard["cross_arch"]])]:
    if len(subset) > 0:
        best = subset.iloc[0]
        print(f"  {label} best: {best['ensemble']}  F1={best['f1']:.4f}")
        print(f"    Mean F1 (top 5): {subset.head(5)['f1'].mean():.4f}")

print()
print("=== Soft Voting: Cross-Arch vs Within-Arch ===")
for label, subset in [("Cross-arch", df_soft[df_soft["cross_arch"]]),
                       ("Within-arch", df_soft[~df_soft["cross_arch"]])]:
    if len(subset) > 0:
        best = subset.iloc[0]
        print(f"  {label} best: {best['ensemble']}  F1={best['f1']:.4f}")
        print(f"    Mean F1 (top 5): {subset.head(5)['f1'].mean():.4f}")

print()
print("=== 2-Model Ensembles: Cross-Arch Advantage ===")
df_hard_2 = df_hard[df_hard["n_models"] == 2]
df_soft_2 = df_soft[df_soft["n_models"] == 2]

for voting, df_v in [("Hard", df_hard_2), ("Soft", df_soft_2)]:
    cross = df_v[df_v["cross_arch"]]["f1"].mean()
    within = df_v[~df_v["cross_arch"]]["f1"].mean()
    print(f"  {voting} voting (2-model): cross-arch mean F1={cross:.4f}, within-arch mean F1={within:.4f}, gap={cross-within:+.4f}")

## 7. Summary: Best Method per Category

In [ ]:
best_single  = df_single.iloc[0]
best_hard    = df_hard.iloc[0]
best_soft    = df_soft.iloc[0]
best_thresh  = df_thresh.iloc[0] if len(df_thresh) > 0 else None

print("=" * 90)
print("SUMMARY: Best Method per Category")
print("=" * 90)
print(f"{'Category':<30s}  {'Method':<40s}  {'F1':>6s}  {'AUC':>6s}")
print("-" * 90)
print(f"{'Single model':<30s}  {best_single['model']:<40s}  {best_single['f1']:.4f}  {best_single['auc']:.4f}")
print(f"{'Hard voting':<30s}  {best_hard['ensemble']:<40s}  {best_hard['f1']:.4f}  {best_hard['auc']:.4f}")
print(f"{'Soft voting':<30s}  {best_soft['ensemble']:<40s}  {best_soft['f1']:.4f}  {best_soft['auc']:.4f}")
if best_thresh is not None:
    print(f"{'Soft + threshold':<30s}  {best_thresh['ensemble']:<40s}  {best_thresh['f1_tuned']:.4f}  {best_thresh['auc']:.4f}")

print()
print("Key insights:")
if best_hard['f1'] > best_single['f1']:
    print(f"  Hard voting vs single: +{best_hard['f1'] - best_single['f1']:.4f} F1")
if best_thresh is not None and best_thresh['f1_tuned'] > best_soft['f1']:
    print(f"  Threshold tuning gain: +{best_thresh['f1_tuned'] - best_thresh['f1_base']:.4f} F1")

## 8. Confusion Matrix — Best Method

In [ ]:
if best_thresh is not None:
    best_models = best_thresh["ensemble"].split("+")
    best_probs = soft_vote_probs(best_models)
    delta_vals = [float(x) for x in best_thresh["delta_str"].strip("()").split(",")]

    log_p = np.log(np.clip(best_probs, 1e-9, 1.0)) + np.array(delta_vals)
    best_preds = log_p.argmax(axis=1)

    cm = confusion_matrix(y_true_ref, best_preds, labels=[0,1,2])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    print(f"Confusion Matrix (normalized) — {best_thresh['ensemble']} + threshold")
    print(f"{'':>15s}  {'Pred Ctrl':>10s}  {'Pred MR':>10s}  {'Pred MS':>10s}")
    for i, name in enumerate(CLASS_NAMES):
        print(f"{name:>15s}  {cm_norm[i,0]:10.3f}  {cm_norm[i,1]:10.3f}  {cm_norm[i,2]:10.3f}   (n={cm[i].sum()})")

    print(f"\nTotal correct: {(best_preds == y_true_ref).sum()} / {len(y_true_ref)} ({(best_preds == y_true_ref).mean():.1%})")